In [1]:
import torch
import torchvision
from torch.utils import data
from torchvision import transforms
from IPython import display

In [2]:
def get_dataloader_workers():
  return 2

In [3]:
def load_data_fashion_mnist(batch_size, resize=None):
    """下载Fashion-MNIST数据集，然后将其加载到内存中"""
    trans = [transforms.ToTensor()]
    if resize:
        trans.insert(0, transforms.Resize(resize))
    trans = transforms.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True,
                            num_workers=get_dataloader_workers()),
            data.DataLoader(mnist_test, batch_size, shuffle=False,
                            num_workers=get_dataloader_workers()))

In [4]:
batch_size = 256
train_iter, test_iter = load_data_fashion_mnist(batch_size)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 200kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.76MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.3MB/s]


In [5]:
num_input = 784
num_output = 10

In [6]:
W = torch.normal(0, 0.01, size=(num_input,num_output), requires_grad=True)
b = torch.zeros(num_output, requires_grad=True)

In [7]:
def softmax(X):
  X_exp = torch.exp(X)
  partition = X_exp.sum(1, keepdim=True)
  return X_exp / partition

In [8]:
def net(X):
  return softmax(torch.matmul(X.reshape((-1, W.shape[0])), W) + b)

In [9]:
def cross_entropy(y_hat, y):
  return -torch.log(y_hat[range(len(y_hat)),y])

In [10]:
def accurarcy(y_hat, y):
  if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
    y_hat = y_hat.argmax(axis=1)
  cmp = y_hat.type(y.dtype) == y
  return float(cmp.type(y.dtype).sum())

In [11]:
class Accumulator:
    """For accumulating sums over `n` variables."""
    def __init__(self, n):
        self.data = [0.0] * n

    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)]

    def reset(self):
        self.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [12]:
def evaluate_accurarcy(net, data_iter):
  if isinstance(net, torch.nn.Module):
    net.eval()
  metric = Accumulator(2)
  for X, y in data_iter:
    metric.add(accurarcy(net(X), y), y.numel())
  return metric[0] / metric[1]

In [13]:
def train_epoch_ch3(net, train_iter, loss, updater):
  if isinstance(net, torch.nn.Module):
    net.train()
  metric = Accumulator(3)
  for X, y in train_iter:
    y_hat = net(X)
    l = loss(y_hat, y)
    if isinstance(updater, torch.optim.Optimizer):
      updater.zero_grad()
      l.backward()
      updater.step()
      metric.add(
          float(1) * len(y), accurarcy(y_hat, y),
          y.size().numel()
      )
    else:
      l.sum().backward()
      updater(X.shape[0])
      metric.add(float(l.sum()), accurarcy(y_hat, y), y.numel())
  return metric[0] / metric[2], metric[1] / metric[2]

In [14]:
def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
  for epoch in range(num_epochs):
    train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
    test_acc = evaluate_accurarcy(net, test_iter)
  train_loss, train_acc = train_metrics

In [15]:
def sgd(params, lr, batch_size):
  """小批量随机梯度下降"""
  with torch.no_grad():
    for param in params:
      param -= lr * param.grad / batch_size
      param.grad.zero_()

In [16]:
lr = 0.1
def updater(batch_size):
  return sgd([W, b], lr, batch_size)

In [17]:
num_epochs = 10
train_ch3(net, train_iter, test_iter, cross_entropy, num_epochs, updater)

/tmp/ipython-input-2093693004.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:835.)
  metric.add(float(l.sum()), accurarcy(y_hat, y), y.numel())
